# DLAI — Full training + all experiments (standalone, no dataset needed)

Notebook **completamente autonomo**: allena tutto da zero e poi esegue tutti gli esperimenti.
Nessun dataset Kaggle da agganciare. Esegui le celle **in ordine dall'alto verso il basso**.

| # | Contenuto |
|---|-----------|
| 1 | Clone repo + install |
| 2 | Crea directory output |
| 3 | GPU check |
| 4 | Train baseline (50 ep, 50k dati) |
| 5 | Train E1-D pairs (30 ep, 2 modelli) |
| 6 | Train E2 models (30 ep, 5 modelli, 10k each) |
| 7 | E2 merging baselines + LMC check |
| 8 | E4 cosine metric + task vector stats |
| 9 | E3 Activation Matching (5 coppie) |
| 10 | E3 REPAIR (5 coppie) + pipeline figure |
| 11 | TIES λ sweep |
| 12 | E6 REPAIR epoch ablation |
| 13 | E8 Greedy soup con val split |
| 14 | E5 Data fraction study (1k/2k/5k) |
| 15 | Display tutti i risultati |
| 16 | Zip finale per download |

## Cella 1 — Clone repo + install package

In [ ]:
import os, subprocess, shutil
REPO_URL = "https://github.com/MarcoLinardi/Deep-Learning-and-Applied-AI-2025-26-exam-project.git"
REPO_DIR = "/kaggle/working/repo"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls
!pip install -q -e .

## Cella 2 — Crea le directory di output

Nessun dataset necessario: tutto viene ri-allenato da zero in questo notebook.

In [ ]:
import os

for d in [
    "results/checkpoints",
    "results/tables",
    "results/plots",
    "results/checkpoints/e5_1k",
    "results/checkpoints/e5_2k",
    "results/checkpoints/e5_5k",
    "results/tables/e5_1k",
    "results/tables/e5_2k",
    "results/tables/e5_5k",
]:
    os.makedirs(d, exist_ok=True)

print("Directory create:")
for root, dirs, _ in os.walk("results"):
    level = root.replace("results", "").count(os.sep)
    print("  " + "  " * level + os.path.basename(root) + "/")

## Cella 3 — GPU check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU non disponibile — abilita il T4 in Settings > Accelerator."
print("device:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Cella 4 — Training baseline ResNet-20

Allena il modello full-data (50 epoche su tutto il train set di CIFAR-10).

In [ ]:
!python -m src.training.train --config configs/baseline.yaml --seed 0 --tag baseline

import pandas as pd
df = pd.read_csv("results/tables/baseline_seed0.csv")
print(f"\nBaseline finale: test_acc = {df['test_acc'].iloc[-1]*100:.2f}%  "
      f"test_loss = {df['test_loss'].iloc[-1]:.4f}")

## Cella 5 — Training coppie E1-D (diff-init, split data)

Allena 2 coppie di modelli con inizializzazioni diverse su dati disgiunti (metà CIFAR-10 ciascuno).
Produce i checkpoint `e1D_pair0_m1.pt` e `e1D_pair0_m2.pt` usati come paia di riferimento in E3.

In [ ]:
!python -m src.experiments.train_pairs --config configs/e1_D_diffinit_splitdata.yaml

import os
ckpts = [f for f in os.listdir("results/checkpoints") if "e1D" in f]
print(f"\nCheckpoint E1-D generati: {sorted(ckpts)}")

## Cella 6 — Training 5 modelli E2 (same-init, split data, 10k each)

Allena 5 modelli ResNet-20 dalla stessa inizializzazione su 5 slice disgiunte da 10k campioni.
Output: `e2_model{0..4}.pt`.

In [ ]:
!python -m src.experiments.train_e2 --config configs/e2_soup.yaml

import os, pandas as pd
print("\nCheckpoint E2 generati:")
for i in range(5):
    path = f"results/tables/e2_model{i}.csv"
    if os.path.isfile(path):
        df = pd.read_csv(path)
        acc = df["test_acc"].iloc[-1] * 100
        print(f"  e2_model{i}: test_acc = {acc:.2f}%")

## Cella 7 — E2 merging baselines + LMC check

Calcola uniform soup, greedy soup, TIES sui 5 modelli E2.
Poi verifica la LMC failure su 3 coppie diagnostiche.

In [ ]:
import subprocess, pandas as pd, os

CWD = "/kaggle/working/repo"

print("=== E2 merging baselines ===")
r = subprocess.run(["python", "-m", "src.experiments.e2_merging",
                    "--config", "configs/e2_soup.yaml"],
                   capture_output=True, text=True, cwd=CWD)
print(r.stdout[-1500:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-500:])

print("\n=== E2 LMC check ===")
r2 = subprocess.run(["python", "-m", "src.experiments.e2_lmc_check",
                     "--config", "configs/e2_soup.yaml"],
                    capture_output=True, text=True, cwd=CWD)
print(r2.stdout[-800:])
if r2.returncode != 0:
    print("STDERR:", r2.stderr[-500:])

print("\n=== E2 aggregate ===")
subprocess.run(["python", "-m", "src.analysis.aggregate_e2",
                "--config", "configs/e2_soup.yaml"],
               capture_output=True, text=True, cwd=CWD)

print(pd.read_csv(os.path.join(CWD, "results/tables/e2_merging.csv")).to_string(index=False))

## Cella 8 — E4 cosine mergeability metric

Calcola cosine similarity tra task vectors delle 10 coppie E2 e la correla con l'accuracy drop al midpoint. Output: `e4_cosine.csv`, `e4_cosine_stats.csv`, `e4_cosine_scatter.png`.

In [ ]:
import subprocess, pandas as pd, os
from IPython.display import Image, display

CWD = "/kaggle/working/repo"

r = subprocess.run(["python", "-m", "src.analysis.mergeability_cosine",
                    "--config", "configs/e2_soup.yaml"],
                   capture_output=True, text=True, cwd=CWD)
print(r.stdout[-800:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-400:])

r2 = subprocess.run(["python", "-m", "src.analysis.task_vector_stats"],
                    capture_output=True, text=True, cwd=CWD)
print(r2.stdout[-600:])
if r2.returncode != 0:
    print("STDERR:", r2.stderr[-400:])

print("\n=== E4 cosine stats ===")
print(pd.read_csv(os.path.join(CWD, "results/tables/e4_cosine_stats.csv")).to_string(index=False))
display(Image(os.path.join(CWD, "results/plots/e4_cosine_scatter.png")))

## Cella 9 — E3 Activation Matching su 5 coppie

Allinea ogni modello B al corrispondente A usando Hungarian assignment su correlazioni di attivazione.
Coppie: `E1D_pair0` (diff-init) + 4 coppie low-resource E2.

In [ ]:
import subprocess, os

CWD = "/kaggle/working/repo"
r = subprocess.run(["python", "-m", "src.experiments.e3_align",
                    "--config", "configs/e3_align.yaml"],
                   capture_output=True, text=True, cwd=CWD)
print(r.stdout[-2000:])
if r.returncode != 0:
    print("ERRORE:", r.stderr[-500:])
else:
    print("\nOK — aligned checkpoints salvati in results/checkpoints/")

## Cella 10 — E3 REPAIR su 5 coppie + figura pipeline

Mini fine-tune 2 epoche lr=1e-3 sul merge `(A + B_aligned)/2`.

In [ ]:
import subprocess, os, pandas as pd
from IPython.display import Image, display

CWD = "/kaggle/working/repo"

# REPAIR
r = subprocess.run(["python", "-m", "src.experiments.e3_repair",
                    "--config", "configs/e3_align.yaml",
                    "--repair-epochs", "2", "--repair-lr", "0.001"],
                   capture_output=True, text=True, cwd=CWD)
print(r.stdout[-1500:])
if r.returncode != 0:
    print("ERRORE:", r.stderr[-500:])

# Pipeline figure (e3_pipeline.png)
r2 = subprocess.run(["python", "-m", "src.analysis.aggregate_final"],
                    capture_output=True, text=True, cwd=CWD)
if r2.returncode != 0:
    print("aggregate_final ERRORE:", r2.stderr[-300:])

print("\n=== E3 REPAIR summary ===")
print(pd.read_csv(os.path.join(CWD, "results/tables/e3_repair.csv")).to_string(index=False))
display(Image(os.path.join(CWD, "results/plots/e3_pipeline.png")))

## Cella 11 — TIES λ sweep

Sweepa λ ∈ {0.3, 0.5, 1.0, 1.5} a keep_ratio=0.20.

In [ ]:
import subprocess, os, pandas as pd

CWD = "/kaggle/working/repo"
r = subprocess.run(["python", "-m", "src.experiments.e2_ties_sweep",
                    "--config", "configs/e2_soup.yaml",
                    "--lams", "0.3", "0.5", "1.0", "1.5"],
                   capture_output=True, text=True, cwd=CWD)
print(r.stdout[-800:])
if r.returncode != 0:
    print("ERRORE:", r.stderr[-500:])

print("\n=== TIES λ sweep results ===")
print(pd.read_csv(os.path.join(CWD, "results/tables/e2_ties_sweep.csv")).to_string(index=False))

## Cella 12 — E6 REPAIR epoch ablation

Esegue 5 epoche di REPAIR sulle 5 coppie di E3 e salva l'accuratezza per ogni epoca.

In [ ]:
import subprocess, os, pandas as pd

CWD = "/kaggle/working/repo"
r = subprocess.run(
    ["python", "-m", "src.experiments.e3_repair",
     "--config", "configs/e3_align.yaml",
     "--repair-epochs", "5", "--repair-lr", "0.001",
     "--history-csv", "results/tables/e6_repair_epochs.csv"],
    capture_output=True, text=True, cwd=CWD
)
print(r.stdout[-2000:])
if r.returncode != 0:
    print("ERRORE:", r.stderr[-500:])

print("\n=== E6 — accuratezza per epoca (pivot) ===")
df6 = pd.read_csv(os.path.join(CWD, "results/tables/e6_repair_epochs.csv"))
print(df6.pivot(index="pair", columns="epoch", values="test_acc")
        .applymap(lambda x: f"{x*100:.2f}%").to_string())

## Cella 13 — E8 Greedy soup con validation split

Usa 5k campioni del training set come val set per la selezione greedy (invece del test set).

In [ ]:
import subprocess, os, csv as _csv, pandas as pd

CWD = "/kaggle/working/repo"
result = subprocess.run(
    ["python", "-m", "src.experiments.e2_merging",
     "--config", "configs/e2_soup.yaml", "--val-split"],
    capture_output=True, text=True, cwd=CWD
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

df_m = pd.read_csv(os.path.join(CWD, "results/tables/e2_merging.csv"))
greedy_val_acc = df_m.loc[df_m["method"] == "greedy_soup", "test_acc"].values[0]
print(f"\nGreedy soup (val-split) test_acc = {greedy_val_acc*100:.2f}%")

out_csv = os.path.join(CWD, "results/tables/e8_greedy_val.csv")
with open(out_csv, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["method", "selection", "test_acc"])
    w.writerow(["greedy_soup", "val_split_5k", greedy_val_acc])
print(f"saved -> {out_csv}")

## Cella 14 — E5 Data fraction study

Addestra 5 modelli × 3 frazioni (1k/2k/5k), applica merging + AM + REPAIR per ogni frazione.

In [ ]:
import subprocess, os

CWD = "/kaggle/working/repo"
FRACS = ["1k", "2k", "5k"]

print("=" * 60)
print("E5a — Training 5 modelli x 3 frazioni")
print("=" * 60)
for frac in FRACS:
    cfg = f"configs/e5_frac{frac}.yaml"
    print(f"\n--- Fraction {frac} ---")
    r = subprocess.run(["python", "-m", "src.experiments.train_e2", "--config", cfg],
                       capture_output=True, text=True, cwd=CWD)
    for l in r.stdout.strip().split("\n")[-5:]:
        print(" ", l)
    if r.returncode != 0:
        print("  ERRORE:", r.stderr[-500:])
    else:
        print(f"  OK — {frac} training completato")

print("\n" + "=" * 60)
print("E5b — Merging baselines x 3 frazioni")
print("=" * 60)
for frac in FRACS:
    cfg = f"configs/e5_frac{frac}.yaml"
    print(f"\n--- Fraction {frac}: merging ---")
    r = subprocess.run(["python", "-m", "src.experiments.e2_merging", "--config", cfg],
                       capture_output=True, text=True, cwd=CWD)
    for l in r.stdout.strip().split("\n")[-6:]:
        print(" ", l)
    if r.returncode != 0:
        print("  ERRORE:", r.stderr[-500:])

In [ ]:
import subprocess, os

CWD = "/kaggle/working/repo"
FRACS = ["1k", "2k", "5k"]

print("=" * 60)
print("E5c — AM alignment × 3 frazioni")
print("=" * 60)
for frac in FRACS:
    cfg = f"configs/e5_align_{frac}.yaml"
    print(f"\n--- Fraction {frac}: AM ---")
    r = subprocess.run(["python", "-m", "src.experiments.e3_align", "--config", cfg],
                       capture_output=True, text=True, cwd=CWD)
    for l in r.stdout.strip().split("\n")[-5:]:
        print(" ", l)
    if r.returncode != 0:
        print("  ERRORE:", r.stderr[-500:])

print("\n" + "=" * 60)
print("E5d — REPAIR × 3 frazioni")
print("=" * 60)
for frac in FRACS:
    cfg = f"configs/e5_align_{frac}.yaml"
    print(f"\n--- Fraction {frac}: REPAIR ---")
    r = subprocess.run(["python", "-m", "src.experiments.e3_repair",
                        "--config", cfg, "--repair-epochs", "2", "--repair-lr", "0.001"],
                       capture_output=True, text=True, cwd=CWD)
    for l in r.stdout.strip().split("\n")[-6:]:
        print(" ", l)
    if r.returncode != 0:
        print("  ERRORE:", r.stderr[-500:])

print("\n" + "=" * 60)
print("E5e — Aggregazione + figura")
print("=" * 60)
r = subprocess.run(["python", "-m", "src.analysis.aggregate_e5"],
                   capture_output=True, text=True, cwd=CWD)
print(r.stdout)
if r.returncode != 0:
    print("ERRORE:", r.stderr[-500:])

## Cella 15 — Display tutti i risultati

In [ ]:
import os, pandas as pd
from IPython.display import Image, display

BASE = "/kaggle/working/repo/results"

def show_csv(path, title):
    full = os.path.join(BASE, path)
    print(f"\n=== {title} ===")
    if os.path.isfile(full):
        print(pd.read_csv(full).to_string(index=False))
    else:
        print(f"  MANCANTE: {full}")

show_csv("tables/e3_repair.csv",           "E3 REPAIR (5 pair)")
show_csv("tables/e3_barriers.csv",         "E3 barriers pre/post AM")
show_csv("tables/e2_ties_sweep.csv",       "TIES λ sweep")
show_csv("tables/e2_task_vector_norms.csv","Task vector norms")
show_csv("tables/e4_cosine_stats.csv",     "E4 cosine stats")
show_csv("tables/e6_repair_epochs.csv",    "E6 REPAIR epoch ablation")
show_csv("tables/e8_greedy_val.csv",       "E8 Greedy val split")
show_csv("tables/e5_summary.csv",          "E5 data fraction summary")

for fig, title in [
    ("plots/e4_cosine_scatter.png", "E4 cosine scatter"),
    ("plots/e3_pipeline.png",       "E3 pipeline"),
    ("plots/e5_fraction.png",       "E5 data fraction curve"),
]:
    full = os.path.join(BASE, fig)
    if os.path.isfile(full):
        print(f"\n--- {title} ---")
        display(Image(full))
    else:
        print(f"\n  {title}: MANCANTE")

## Cella 16 — Zip finale per download

Raccoglie tutti i CSV e plot in un file zip. Da scaricare e riportare nella repo.

In [ ]:
import os, glob, subprocess

BASE = "/kaggle/working/repo/results"
files = (
    glob.glob(f"{BASE}/tables/*.csv") +
    glob.glob(f"{BASE}/tables/e5_*/*.csv") +
    glob.glob(f"{BASE}/plots/*.png") +
    glob.glob(f"{BASE}/checkpoints/e3_E2_*_merged_repaired.pt") +
    glob.glob(f"{BASE}/checkpoints/e3_E2_*_m2_aligned.pt")
)
files = sorted(f for f in files if os.path.isfile(f))
print(f"Totale file: {len(files)}")
for f in files:
    print(" ", f.replace("/kaggle/working/repo/", ""))

zip_path = "/kaggle/working/dlai_FINAL_outputs.zip"
r = subprocess.run(["zip", zip_path] + files, capture_output=True, text=True)
if r.returncode == 0:
    print(f"\nOK: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
    print("Scarica dal tab Output (colonna destra di Kaggle).")
else:
    print("ERRORE:", r.stderr)